# Capstone: the charity shop sorting machine

MichAl Academy, lesson 3.14.

A charity shop takes donations through a bin at the back. Somebody has to open
every bag and sort what is in it onto the right rail: trousers here, coats there,
shoes on the shelf. They have asked for a machine that looks at a photograph of
one item and says which rail it goes on.

You get **Fashion-MNIST**: 70,000 photographs of clothing, 28 by 28 pixels,
greyscale, in ten categories. It was built as a drop-in replacement for the
handwritten digits, because digits had become too easy to be interesting.

**Your job is not to get the highest score.** It is to end up able to answer one
question honestly: *which rail would you let this machine sort unsupervised, and
which would you still have a person check?*

Work the tasks in order. Each has a check cell that tells you when you have it.
Answers are folded at the bottom, and reading them early wastes the exercise.

Budget about ninety minutes.


In [ ]:
import time
import warnings

import numpy as np
import torch
from sklearn.datasets import fetch_openml
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")
torch.set_num_threads(1)

NAMES = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
         "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]

data = fetch_openml("Fashion-MNIST", version=1, as_frame=False)
images = data.data.astype("float32") / 255.0
labels = data.target.astype(int)

rng = np.random.default_rng(0)
order = rng.permutation(len(images))
train_idx, test_idx = order[:8000], order[8000:12000]

X_train, y_train = images[train_idx], labels[train_idx]
X_test, y_test = images[test_idx], labels[test_idx]

X_test_t = torch.tensor(X_test)
y_test_t = torch.tensor(y_test)

print(f"{len(X_train)} training photographs, {len(X_test)} held back")
print(f"each one is {images.shape[1]} numbers between 0 and 1")
print(f"ten rails: {', '.join(NAMES)}")


## The two helpers

`fit` trains a network and `accuracy` scores it on the held-back photographs.
Both are used by every task below, so read them once now.

Every task seeds the weights immediately before building the network, so each
one gives the same answer whether you run the notebook top to bottom or re-run a
single cell. If a result changes, it is because you changed something.

That matters more than it sounds. Without it, a network's starting weights depend
on how much of the notebook has already run, and task 4 alone moved by 0.28 on
one category between two orderings.


In [ ]:
def fit(model, X, y, epochs, lr=0.001, batch_size=64, seed=0,
        optimiser="adam", conv=False):
    """Train `model` and return it. `conv=True` reshapes flat rows into images."""
    torch.manual_seed(seed)
    make = torch.optim.Adam if optimiser == "adam" else torch.optim.SGD
    opt = make(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()

    rows = torch.tensor(X)
    if conv:
        rows = rows.view(-1, 1, 28, 28)
    loader = DataLoader(TensorDataset(rows, torch.tensor(y)),
                        batch_size=batch_size, shuffle=True)

    for _ in range(epochs):
        for batch_X, batch_y in loader:
            opt.zero_grad()
            loss_fn(model(batch_X), batch_y).backward()
            opt.step()
    return model


def predictions(model, conv=False):
    """What the model says about each held-back photograph."""
    model.eval()
    rows = X_test_t.view(-1, 1, 28, 28) if conv else X_test_t
    with torch.no_grad():
        out = model(rows).argmax(dim=1)
    model.train()
    return out


def accuracy(model, conv=False):
    return float((predictions(model, conv) == y_test_t).float().mean())


## Task 1: get a baseline on the board

Build the simplest thing that could work: 784 inputs, one hidden layer of 128
units with a ReLU, ten outputs. Train it for 15 epochs.

Before you run it, write down what accuracy you expect. Ten rails means guessing
scores 0.10.

Record the accuracy you get in `baseline`.


In [ ]:
started = time.time()

torch.manual_seed(0)
dense = nn.Sequential(
    nn.Linear(784, 128),
    nn.ReLU(),
    nn.Linear(128, 10),
)
fit(dense, X_train, y_train, epochs=15)

print(f"accuracy: {accuracy(dense):.4f}   ({time.time() - started:.1f}s)")


In [ ]:
baseline = None   # TODO: the accuracy the cell above printed

print("task 1 done?", baseline is not None and baseline > 0.80)


## Task 2: find out which rail it cannot handle

A single accuracy is an average over ten rails, and the shop does not experience
an average. They experience the rail where the machine is wrong.

Score each category separately, then find which categories are being confused
with which. Record the name of the worst category in `worst_rail`.


In [ ]:
pred = predictions(dense)

print("accuracy on each rail, worst first")
per_class = {}
for c in range(10):
    mask = torch.tensor(y_test == c)
    per_class[NAMES[c]] = float((pred[mask] == c).float().mean())

for name, score in sorted(per_class.items(), key=lambda kv: kv[1]):
    print(f"  {name:14s} {score:.4f}")

print("\nmost common mistakes")
confusions = []
for a in range(10):
    for b in range(10):
        if a != b:
            n = int(((y_test == a) & (pred.numpy() == b)).sum())
            confusions.append((n, NAMES[a], NAMES[b]))

for n, actual, guessed in sorted(confusions, reverse=True)[:5]:
    print(f"  {actual:14s} called {guessed:14s} {n} times")


In [ ]:
worst_rail = None   # TODO: the name of the worst category, exactly as printed

print("task 2 done?", worst_rail in NAMES and per_class[worst_rail] == min(per_class.values()))


## Task 3: the setting that decides whether anything is learned

Lesson 3.5 said the learning rate is the one setting worth getting right, and
that both failures are silent. Here are four combinations. Two of them work and
two do not, and neither failure raises an error.

Predict which two will fail before you run it. Then record the best accuracy you
saw in `best_setting_accuracy`.


In [ ]:
for optimiser, lr in [("sgd", 0.01), ("sgd", 0.5), ("adam", 0.1), ("adam", 0.001)]:
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
    fit(model, X_train, y_train, epochs=8, lr=lr, optimiser=optimiser)
    print(f"{optimiser:5s} lr={lr:<7} {accuracy(model):.4f}")


In [ ]:
best_setting_accuracy = None   # TODO: the best of the four numbers above

print("task 3 done?", best_setting_accuracy is not None and best_setting_accuracy > 0.82)


## Task 4: give the network the fact that these are pictures

The dense network throws away the shape of the image: it sees 784 unrelated
numbers and has to learn from scratch that pixel 1 and pixel 2 are neighbours.
A convolutional network is told.

Same data, same optimiser, same learning rate. Twelve epochs this time, because
a convolutional network on a CPU takes more passes to settle.

This cell takes about half a minute. When it finishes, compare it with task 1
both overall **and** on the rail you found in task 2.


In [ ]:
started = time.time()

torch.manual_seed(0)
cnn = nn.Sequential(
    nn.Conv2d(1, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(),
    nn.Linear(32 * 7 * 7, 64), nn.ReLU(),
    nn.Linear(64, 10),
)
fit(cnn, X_train, y_train, epochs=12, conv=True)

cnn_pred = predictions(cnn, conv=True)
cnn_overall = accuracy(cnn, conv=True)

print(f"overall     dense {accuracy(dense):.4f}   cnn {cnn_overall:.4f}")
for c in range(10):
    mask = torch.tensor(y_test == c)
    d = float((pred[mask] == c).float().mean())
    k = float((cnn_pred[mask] == c).float().mean())
    flag = "  <-- worst rail" if d == min(per_class.values()) else ""
    print(f"{NAMES[c]:14s} dense {d:.4f}   cnn {k:.4f}   {k - d:+.4f}{flag}")

print(f"\n({time.time() - started:.1f}s)")


In [ ]:
cnn_beat_dense = None   # TODO: True or False, overall

print("task 4 done?", cnn_beat_dense is (cnn_overall > baseline) if baseline else False)


## Task 5: make more photographs out of the ones you have

The shop will photograph items any way round. A jumper laid down facing left is
the same jumper facing right, so a mirrored copy of every training photograph is
a free extra example, and clothing is roughly symmetric left to right.

Double the training set with horizontal flips and train the same dense network
on twice as much data.

**Expect this to help, and check whether it did.** Record what you find in
`flips_helped`.


In [ ]:
flipped = np.flip(X_train.reshape(-1, 28, 28), axis=2).reshape(-1, 784)
X_aug = np.concatenate([X_train, flipped])
y_aug = np.concatenate([y_train, y_train])

torch.manual_seed(0)
augmented = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
fit(augmented, X_aug, y_aug, epochs=15)

aug_score = accuracy(augmented)
print(f"{len(X_train)} photographs  {accuracy(dense):.4f}")
print(f"{len(X_aug)} photographs  {aug_score:.4f}")
print(f"difference             {aug_score - accuracy(dense):+.4f}")


In [ ]:
flips_helped = None   # TODO: True or False

print("task 5 done?", flips_helped is (aug_score > accuracy(dense)))


## Task 6: the rail you have almost no labels for

A new category arrives and somebody has hand-labelled fifty photographs of it.
Fifty is what you get.

Here that is the hardest pair in the whole dataset: pullover against shirt. Two
networks are trained on the same fifty labels. One starts from random weights.
The other starts from a network trained on the eight *other* categories, which
has never seen a pullover or a shirt.

Everything else is identical, which is the only way the comparison means
anything.


In [ ]:
import copy

SOURCE = [0, 1, 3, 4, 5, 7, 8, 9]
renumber = {c: i for i, c in enumerate(SOURCE)}

src = np.isin(y_train, SOURCE)
torch.manual_seed(0)
source_net = nn.Sequential(
    nn.Linear(784, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 8),
)
fit(source_net, X_train[src], np.array([renumber[v] for v in y_train[src]]), epochs=12)

pick = np.isin(y_train, [2, 6])
X_hard, y_hard = X_train[pick], (y_train[pick] == 6).astype(int)
held = np.isin(y_test, [2, 6])
X_hard_test = torch.tensor(X_test[held])
y_hard_test = torch.tensor((y_test[held] == 6).astype(int))

draw = np.random.default_rng(0)
fifty = np.concatenate([draw.choice(np.flatnonzero(y_hard == c), 25, replace=False)
                        for c in (0, 1)])

def hard_score(model):
    model.eval()
    with torch.no_grad():
        out = model(X_hard_test).argmax(dim=1)
    model.train()
    return float((out == y_hard_test).float().mean())

torch.manual_seed(1)
scratch = nn.Sequential(nn.Linear(784, 128), nn.ReLU(),
                        nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 2))
fit(scratch, X_hard[fifty], y_hard[fifty], epochs=30, batch_size=16)

transferred = copy.deepcopy(source_net)
torch.manual_seed(1)
transferred[4] = nn.Linear(64, 2)
fit(transferred, X_hard[fifty], y_hard[fifty], epochs=30, batch_size=16)

print(f"50 labels, from random weights    {hard_score(scratch):.4f}")
print(f"50 labels, from the source net    {hard_score(transferred):.4f}")


In [ ]:
transfer_helped = None   # TODO: True or False

print("task 6 done?", transfer_helped is (hard_score(transferred) > hard_score(scratch)))


## The question

You now have six measurements. The shop does not want any of them. They want an
answer to this:

**Which rails would you let the machine sort unsupervised, and which would you
still have a person check?**

Write it below, in the cell, in your own words. Three sentences is plenty. Name
the rails, give the number that justifies each decision, and say what you would
measure next.

There is no check cell for this one. It is the only task that matters.


In [ ]:
verdict = """
TODO: your answer.
"""
print(verdict)


## Optional: beat the machine

Everything above is a floor, not a ceiling. Things worth trying, roughly in
order of how much they usually buy:

- Train the convolutional network for longer than twelve epochs.
- Use all 70,000 photographs instead of the 12,000 this notebook samples.
- Add a third convolutional block, or widen the ones that are there.
- Augment with small shifts and rotations rather than flips.
- Look at the photographs the model gets wrong. Some of them are genuinely
  ambiguous, and that puts a ceiling on any model.


## Answers

Do not read these until you have your own.

<details>
<summary>Task 1: the baseline</summary>

**0.8553.** Ten rails means guessing scores 0.10, so the simplest network anyone
would write is already doing most of the job. That is worth noticing before
reaching for anything more complicated, and it is why lesson 2.2 asks for the
cheap baseline first.

</details>

<details>
<summary>Task 2: the worst rail</summary>

**Shirt, at 0.5787**, against Trouser at 0.9791. The gap between the best rail
and the worst is larger than the gap between this model and a very good one.

The mistakes say why:

| | |
|---|---|
| Shirt called Pullover | 72 |
| Pullover called Coat | 54 |
| Coat called Pullover | 47 |
| T-shirt/top called Shirt | 45 |
| Shirt called T-shirt/top | 45 |

Every one of those is an upper-body garment with sleeves, photographed flat at
28 by 28. The machine is not confused about clothing in general. It is confused
about one cluster of four categories, and an accuracy of 0.8553 hid that
completely.

</details>

<details>
<summary>Task 3: the learning rate</summary>

| | |
|---|---|
| SGD, 0.01 | 0.7492 |
| SGD, 0.5 | 0.8155 |
| Adam, 0.1 | 0.4505 |
| Adam, 0.001 | 0.8403 |

Both failures are the ones lesson 3.5.1 describes, and neither raises an error.
SGD at 0.01 takes steps too small to arrive in eight epochs, so it simply stops
at 0.7492. Adam at 0.1 takes steps too large and lands at 0.4505: a model that
works, is badly worse than it should be, and says nothing about it in the log.

Note that the two optimisers want learning rates that differ by a factor of
about five hundred. **A learning rate is not transferable between optimisers**,
and carrying one over is a common way to conclude that an optimiser is bad.

</details>

<details>
<summary>Task 4: the convolutional network</summary>

**0.8680 against the dense network's 0.8553**, so structure won. And it won in
the way lesson 3.8.2 predicts: Coat improves by 0.1235, Sneaker by 0.0769,
T-shirt/top by 0.0657. Those are categories separated by shape, and convolution
is the architecture that recognises a shape wherever it sits in the frame.

**Now look at the Shirt rail: 0.5787 down to 0.4625.** The best model in the
notebook is substantially worse at the one category that was already worst.

This is not a fluke of one run. Repeating the whole comparison with six
different starting seeds, the convolutional network wins overall in **6 of 6**,
by between 0.0067 and 0.0127, and loses the Shirt rail in **6 of 6**, by between
0.1162 and 0.1961.

So the model got better and the shop's problem got worse. Shirt is the category
with no distinctive shape at all, and a network that has learned to lean on
shape has less to go on there than one that looked at every pixel equally. The
overall number went up because the other nine rails carried it.

**A trap on the way.** At six epochs instead of twelve this network scores about
0.83 and its Shirt accuracy swings between 0.51 and 0.75 depending on the seed.
An undertrained model is not a worse architecture, and comparing two
architectures at a budget that suits only one of them is not a comparison.

</details>

<details>
<summary>Task 5: the flips</summary>

**It did not help: 0.8505 against 0.8553.** The reasoning was sound and the
result was slightly negative.

Augmentation pays when it shows the model variation it will meet and has not
already seen. These photographs are centred and upright, and the garments are
close enough to left-right symmetric that a mirrored jumper is one the model has
effectively seen already. Doubling the rows doubled the training time and added
no information.

That is the ordinary shape of an augmentation result, and it is why
lesson 3.8.4 says to measure it rather than assume it. Small shifts would be the
better bet here, because the donation bin really will produce off-centre
photographs and the mirror really will not.

</details>

<details>
<summary>Task 6: transfer</summary>

**0.7754 from the source network against 0.7618 from random weights**, on fifty
labels, for the hardest pair in the set.

The gain is small, and it came from a source network that saw 6,400 small
greyscale photographs. The models used for this in practice saw millions, which
is what lesson 3.13.4 is about.

The direction is the part to keep, and so is the price. The source network was
trained on categories you already had labels for, and reusing it was one line.

</details>

<details>
<summary>The question</summary>

There is no single right answer, and a good one names rails and numbers rather
than describing the model. Something with this shape:

*Trouser (0.9791), Ankle boot (0.9751), Bag (0.9616) and Sandal (0.9241) can go
unsupervised. The upper-body rail, meaning shirt, pullover, coat and
T-shirt/top, needs a person: shirt is 0.5787 with the dense network and 0.4625
with the convolutional one, and its errors land on the three categories next to
it rather than spreading out. I would ship the dense model rather than the more
accurate convolutional one, because the extra 0.0127 overall is not worth 0.1162
on the rail that was already the problem. Before anything else I would
photograph two hundred real donations and check these numbers hold, because
every one of them is on clean centred images and a donation bin does not produce
those.*

Two things separate that from a score. It picks the worse model on purpose and
says why, which is what lesson 2.14 means by choosing the metric that matches
the cost. And it distrusts its own numbers, which is lesson 2.23: every
measurement here is on a tidy benchmark, and the shop is not a tidy benchmark.

</details>
